In [ ]:
#!pip uninstall tft_pytorch -y

In [ ]:
#!pip install -U --no-cache git+https://github.com/rsscml/tft_pytorch

In [ ]:
#!pip install -U openpyxl

In [ ]:
# base imports
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import time
import copy
import os
import random
import warnings
from joblib import Parallel, delayed

# import TFT
import torch
from tft_pytorch import (
    OptimizedTFTDataset, 
    create_tft_dataloader, 
    create_uniform_embedding_dims,
    TemporalFusionTransformer,
    TFTTrainer,
    TFTInferenceWithTracking)

%matplotlib inline
pd.set_option("display.max_columns", 1000)
pd.set_option("display.max_rows", 1000)
warnings.filterwarnings("ignore")


In [ ]:
DATA_PATH = "../data/plant_mat_channel_agg_input.xlsx"

In [ ]:
df = pd.read_excel(DATA_PATH, dtype={"Plant": 'str', "Material": 'str', "Channel": 'str'})

# additional feats
df['Month'] = df['YearMonth'].dt.month
df['Quarter'] = df['YearMonth'].dt.quarter

# power transform
df['Customer Demand Qty'] = np.sqrt(df['Customer Demand Qty'])

print(df.groupby(['Plant'])['key'].nunique())


In [ ]:
df.head()

In [ ]:
# BG Plant

df = df[df['Plant']=='3891']

df.shape

In [ ]:

# features config
features_config = {
    "entity_col": "key",
    "time_index_col": "YearMonth",
    "target_col": "Customer Demand Qty",

    # known in the future
    "temporal_known_numeric_col_list": ["Working_Days"],
    "temporal_known_categorical_col_list": ["Month", "Quarter","Minimum Lot Size"],

    # only historical
    "temporal_unknown_numeric_col_list": [],
    "temporal_unknown_categorical_col_list": [],

    # static per entity
    "static_numeric_col_list": [],
    "static_categorical_col_list": ["Material","Channel","Material_Group","Material_Hierarchy"]
}

# context window length
historical_steps=24
forecast_steps=3
train_min_historical_steps=12
test_min_historical_steps=12
infer_min_historical_steps=0
test_periods=6

# no. of samples
train_stride=1
test_stride=1

# parallel processing
train_n_jobs = 4
test_n_jobs = 4
infer_n_jobs = 1

# scaler and encoder location (overwritten for each eval run)
scaler_path='./artefacts/train_scalers.pkl'
scaling_strategy='entity_level'
scaling_method='mean'
encoders_path='./artefacts'

# train cutoff points for various eval runs
ORIGINS = [
    (pd.Timestamp("2024-07-01"), pd.Timestamp("2025-01-01"), pd.Timestamp("2025-04-01")), # -> target April-2025
    (pd.Timestamp("2024-08-01"), pd.Timestamp("2025-02-01"), pd.Timestamp("2025-05-01")), # -> target May-2025
    (pd.Timestamp("2024-09-01"), pd.Timestamp("2025-03-01"), pd.Timestamp("2025-06-01")), # -> target Jun-2025
    (pd.Timestamp("2024-10-01"), pd.Timestamp("2025-04-01"), pd.Timestamp("2025-07-01")), # -> target Jul-2025
    (pd.Timestamp("2024-11-01"), pd.Timestamp("2025-05-01"), pd.Timestamp("2025-08-01")), # -> target Aug-2025
    (pd.Timestamp("2024-12-01"), pd.Timestamp("2025-06-01"), pd.Timestamp("2025-09-01")), # -> target Sep-2025
    (pd.Timestamp("2025-01-01"), pd.Timestamp("2025-07-01"), pd.Timestamp("2025-10-01")), # -> target Oct-2025
    (pd.Timestamp("2025-02-01"), pd.Timestamp("2025-08-01"), pd.Timestamp("2025-11-01")), # -> target Nov-2025
    (pd.Timestamp("2025-03-01"), pd.Timestamp("2025-09-01"), pd.Timestamp("2025-12-01")), # -> target Dec-2025
]


In [ ]:
# Eval loop

lag3_eval_df_list = []

for i, origin in enumerate(ORIGINS):

    print(f"Eval run: {i+1}")
    
    # ----------------- split dataframe ------------------------
    train_start = pd.to_datetime('2020-01-01', format="%Y-%m-%d")
    train_cutoff = origin[0]
    
    test_start = train_cutoff - pd.DateOffset(months=historical_steps)
    test_cutoff = origin[1]

    infer_cutoff = origin[2]
    infer_start = infer_cutoff - pd.DateOffset(months=historical_steps + forecast_steps)

    print(f" train_start: {train_start}, train_cutoff: {train_cutoff}")
    print(f" test_start: {test_start}, test_cutoff: {test_cutoff}")
    print(f" infer_start: {infer_start}, infer_cutoff: {infer_cutoff}")
    
    train_df = df[df['YearMonth'] <= train_cutoff].copy()
    test_df = df[(df['YearMonth'] > test_start) & (df['YearMonth'] <= test_cutoff)].copy()
    
    infer_df = df[df['YearMonth'] <= infer_cutoff].copy()
    infer_df =  infer_df.groupby(['key'], group_keys=False).tail(historical_steps + forecast_steps)
    
    print(train_df.shape, test_df.shape, infer_df.shape)

    # ------------------ create datasets ----------------------------
    train_dataset = OptimizedTFTDataset(
                        data_source=train_df,
                        features_config=features_config, 
                        historical_steps=historical_steps,
                        prediction_steps=forecast_steps,
                        stride=train_stride,
                        enable_padding=True,
                        padding_strategy = 'zero',
                        categorical_padding_value = -1,
                        min_historical_steps=train_min_historical_steps,
                        allow_inference_only_entities=False,
                        scaler_path=scaler_path,
                        scaling_strategy=scaling_strategy,
                        scaling_method=scaling_method,
                        cold_start_scaler_cols=None,
                        mean_scaler_epsilon=1.0,
                        recency_alpha=1.0,
                        n_jobs=train_n_jobs,
                        max_series=None,
                        mode='train',
                        encoders_path=encoders_path,
                        fit_encoders=None,
                        preprocessing_fn=None)
    
    test_dataset = OptimizedTFTDataset(
                        data_source=test_df,
                        features_config=features_config, 
                        historical_steps=historical_steps,
                        prediction_steps=forecast_steps,
                        stride=test_stride,
                        enable_padding=True,
                        padding_strategy = 'zero',
                        categorical_padding_value = -1,
                        min_historical_steps=test_min_historical_steps,
                        allow_inference_only_entities=False,
                        scaler_path=scaler_path,
                        scaling_strategy=scaling_strategy,
                        scaling_method=scaling_method,
                        cold_start_scaler_cols=None,
                        mean_scaler_epsilon=1.0,
                        recency_alpha=0,
                        n_jobs=test_n_jobs,
                        max_series=None,
                        mode='test',
                        encoders_path=encoders_path,
                        fit_encoders=None,
                        preprocessing_fn=None)

    infer_dataset = OptimizedTFTDataset(
                        data_source=infer_df,
                        features_config=features_config, 
                        historical_steps=historical_steps,
                        prediction_steps=forecast_steps,
                        stride=test_stride,
                        enable_padding=True,
                        padding_strategy = 'zero',
                        categorical_padding_value = -1,
                        min_historical_steps=infer_min_historical_steps,
                        allow_inference_only_entities=True,
                        scaler_path=scaler_path,
                        scaling_strategy=scaling_strategy,
                        scaling_method=scaling_method,
                        cold_start_scaler_cols=['Material','Channel'], 
                        mean_scaler_epsilon=1.0,
                        recency_alpha=0,
                        n_jobs=1,
                        max_series=None,
                        mode='test',
                        encoders_path=encoders_path,
                        fit_encoders=None,
                        preprocessing_fn=None)

    # ------------------------ create dataloaders -------------------------------
    train_dataloader, train_adapter = create_tft_dataloader(train_dataset, batch_size=128, shuffle=True, num_workers=0, drop_last=False, pin_memory=True)

    test_dataloader, test_adapter = create_tft_dataloader(test_dataset, batch_size=128, shuffle=False, num_workers=0, drop_last=False, pin_memory=True)

    infer_dataloader, infer_adapter = create_tft_dataloader(infer_dataset, batch_size=128, shuffle=False, num_workers=0, drop_last=False, pin_memory=True)

    # ------------------- create embedding dims dict for various categorical variables
    categorical_embedding_dims = create_uniform_embedding_dims(train_dataset, hidden_layer_size=160)
    num_static_categorical=len(train_dataset.static_categorical_cols)
    num_static_continuous=len(train_dataset.static_numeric_cols)
    num_historical_categorical=len(train_dataset.temporal_unknown_categorical_cols) + len(train_dataset.temporal_known_categorical_cols)
    num_historical_continuous=len(train_dataset.target_cols) + len(train_dataset.temporal_unknown_numeric_cols) + len(train_dataset.temporal_known_numeric_cols)
    num_future_categorical=len(train_dataset.temporal_known_categorical_cols)
    num_future_continuous=len(train_dataset.temporal_known_numeric_cols)

    # ---------------------- create model -----------------------------------
    model = TemporalFusionTransformer(
            # Model architecture parameters
            hidden_layer_size = 160,
            num_attention_heads = 1,
            num_lstm_layers = 1,
            num_attention_layers = 4,
            dropout_rate = 0.1,
            
            # Input specification
            num_static_categorical = num_static_categorical,
            num_static_continuous = num_static_continuous,
            num_historical_categorical = num_historical_categorical,
            num_historical_continuous = num_historical_continuous,
            num_future_categorical = num_future_categorical,
            num_future_continuous = num_future_continuous,
        
            # embedding dims for categorical variables
            categorical_embedding_dims = categorical_embedding_dims,
            
            # Time dimensions
            historical_steps = historical_steps,
            prediction_steps = forecast_steps,
            
            # Output specification
            num_outputs = 1,
            device = 'cuda' #'cuda' if torch.cuda.is_available() else 'cpu'
        )
    
    # ------------------- create trainer ------------------------------------
    trainer = TFTTrainer(model = model,
                     # data params
                     train_loader = train_dataloader,
                     val_loader = test_dataloader,
                     train_adapter = train_adapter,
                     val_adapter = test_adapter,
                     # loss params
                     loss_type = 'huber',
                     loss_params = {'delta': 1.0},
                     # optimizer params
                     optimizer_type = 'adam',
                     learning_rate = 0.0001,
                     # Scheduler configuration
                     scheduler_type = 'reduce_on_plateau',
                     scheduler_factor = 0.5,
                     scheduler_patience = 5,
                     # Training options
                     enable_gradient_clipping = True,
                     max_grad_norm = 1.0,
                     enable_train_sample_weighting = False,
                     enable_val_sample_weighting = False,
                     # Mixed Precision Training
                     enable_mixed_precision = False,
                     # Checkpointing
                     save_path = './models/china_plant_mat_ch',
                     save_every = 1)
    
    # ------------------------ train -----------------------------------------
    trainer.train(num_epochs=50, patience=10)

    # ----------------------- infer ------------------------------------------
    inferencewithtracking = TFTInferenceWithTracking(model_path = 'models/china_plant_mat_ch/best_model.pt', model = model, adapter = infer_adapter, device = 'cuda')
    results_df = inferencewithtracking.predict_with_metadata(infer_dataloader)

    # ----------------------- inverse power transform -------------------------
    results_df['pred_0'] = np.square(results_df['pred_0'])
    results_df['actual_Customer Demand Qty'] = np.square(results_df['actual_Customer Demand Qty'])

    # get lag3
    lag3_results_df = results_df[results_df['timestamp']==origin[2]]
    
    lag3_eval_df_list.append(lag3_results_df)

    # ----------------------- clear model ------------------------------------- 
    del model
    gc.collect()



print("Combine & save results")
results_df = pd.concat(lag3_eval_df_list, ignore_index=True)

# Save
results_df.to_csv("./outputs/ch_huber_1_transform_plant_material_channel_lag3eval_bg.csv", index=False)
